# 02 · Baseline models

Chronological split only — the first 3 000 records train the initial champion (80/20 inside that
period for validation). No shuffling of the time series.

In [1]:
import pandas as pd
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 20)

from sentinelml.common.config import get_settings
from sentinelml.ingestion.loaders import load_stream
from sentinelml.models.train import evaluate, latency_profile, train_model
from sentinelml.validation.quality import clean_batch

s = get_settings()
df = load_stream("sudden", seed=0)
train_period = clean_batch(df[df.batch_id < 12], s.data["schema"])
cut = int(len(train_period) * .8)
tr, va = train_period.iloc[:cut], train_period.iloc[cut:]
print(len(tr), "train /", len(va), "validation")

2400 train / 600 validation


## Majority baseline vs. three model families

In [2]:
rows = [{"model": "majority_class", "roc_auc": 0.5, "accuracy": max(va.label.mean(), 1 - va.label.mean())}]
for algo in ["logistic_regression", "random_forest", "hist_gradient_boosting"]:
    m = train_model(tr, algo)
    rows.append({"model": algo, **evaluate(m, va), **latency_profile(m, va)})
res = pd.DataFrame(rows).set_index("model").round(3)
res

,roc_auc,accuracy,precision,recall,f1,pr_auc,latency_ms_per_1k,p50_ms,p95_ms
model,,,,,,,,,
majority_class,0.500,0.518,NaN,NaN,NaN,NaN,NaN,NaN,NaN
logistic_regression,0.937,0.858,0.867,0.859,0.863,0.944,10.399,5.093,5.776
random_forest,0.915,0.833,0.860,0.810,0.834,0.922,105.066,50.171,53.074
hist_gradient_boosting,0.910,0.830,0.852,0.814,0.832,0.917,23.619,6.792,9.428


**Decision:** on this stream logistic regression has the best validation ROC-AUC, the lowest
single-record latency (random forest is ~8× slower) and exact additive explanations — so it is the
default champion family (`configs/model.yaml`). Tree models remain one config change away and are
worth re-testing on real data with non-linear effects.